# Chapter 5: JAX — High-Performance Numerical Computing for Research

### Why This Matters

JAX (from Google) is less a "deep learning framework" in the Keras/PyTorch sense and more a high-performance, composable foundation: a NumPy-compatible API (recall Volume 02) that adds automatic differentiation and just-in-time (JIT) compilation to accelerator hardware (GPU/TPU). It's heavily used in cutting-edge research and increasingly in production systems that need custom, highly optimized numerical code beyond what pre-built layer APIs offer.

In [ ]:
import jax
import jax.numpy as jnp

# Automatic differentiation of an arbitrary Python function
f = lambda x: x**3 + 2*x
grad_f = jax.grad(f)          # returns a NEW function computing the derivative
print(grad_f(3.0))            # 3*(3^2) + 2 = 29.0

# JIT compilation — compiles a function to run much faster on repeated calls
@jax.jit
def fast_fn(x):
    return jnp.sin(x) + x**2

`jax.grad(f)` is conceptually similar to PyTorch's `.backward()`, but framed differently: rather than calling a method on a tensor after building a graph imperatively, `jax.grad` takes an ordinary Python function and *returns a new function* that computes its derivative — a more explicitly "functional programming" style than PyTorch's object-oriented one. `@jax.jit` is JAX's signature performance feature: it compiles a Python function (the first time it's called) into highly optimized machine code specialized for your input shapes, so subsequent calls run dramatically faster — particularly valuable for tight numerical loops (physics simulations, custom optimization algorithms, research architectures) where the overhead of ordinary interpreted Python would otherwise dominate.

💡 **`vmap`** — JAX's automatic vectorization transformation — deserves a mention even though it's beyond this chapter's depth: it takes a function written for a *single* example and automatically produces a version that efficiently runs over a whole *batch* of examples, without you manually rewriting the function to handle an extra batch dimension. This "write it simply once, vectorize it automatically" philosophy runs through JAX's whole design.

⚠️ **Practical guidance:** unless your work specifically involves research-style custom architectures, novel training algorithms, or you're joining a team already standardized on JAX, you're unlikely to need it directly for typical business analytics work — Keras and PyTorch (with or without Lightning) comfortably cover the large majority of practical business deep learning needs. Treat this chapter as "know it exists and roughly what it's for" rather than a tool to reach for by default.

### Cheat Sheet — JAX

| Task | Code |
|---|---|
| NumPy-like ops | `import jax.numpy as jnp` |
| Differentiate a function | `jax.grad(f)` |
| Compile for speed | `@jax.jit` |
| Auto-vectorize | `jax.vmap(f)` |

---

## Choosing a Deep Learning Framework: A Decision Guide

| Situation | Reach for |
|---|---|
| Learning deep learning for the first time | **Keras** |
| You want maximum flexibility / custom architectures | **PyTorch** |
| A serious, reproducible project needing clean engineering | **PyTorch Lightning** |
| Fast results with minimal code, especially computer vision | **FastAI** |
| Research-grade custom numerics, or joining a JAX-based team | **JAX** |
| The problem is tabular business data | **Stop — go back to Volume 06's gradient boosting first** |

### 🎯 Business Challenge — Volume 08

*Task (from the source material):* build a simple neural network on the churn dataset and compare it honestly against your Volume 06 LightGBM/XGBoost baseline.

In [ ]:
import tensorflow as tf
from tensorflow import keras
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

X_scaled = StandardScaler().fit_transform(X_train)
X_test_scaled = StandardScaler().fit(X_train).transform(X_test)     # NOTE: fit scaler on train only (Volume 05)

nn_model = keras.Sequential([
    keras.layers.Dense(32, activation="relu", input_shape=(X_train.shape[1],)),
    keras.layers.Dropout(0.2),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.Dense(1, activation="sigmoid")
])
nn_model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
nn_model.fit(X_scaled, y_train, validation_split=0.2, epochs=30, batch_size=32, verbose=0)

nn_proba = nn_model.predict(X_test_scaled).ravel()
print("Neural Network ROC-AUC:", round(roc_auc_score(y_test, nn_proba), 4))
print("(Compare this against your LightGBM/XGBoost ROC-AUC from Volume 06's same dataset)")

*Discussion prompt to actually answer, not skip:* Did the neural network beat the gradient boosting baseline? If not (the common outcome on tabular data of this size), was the additional training time, tuning effort, and reduced interpretability worth it? For most tabular business problems at this scale, the honest answer is no — which is itself the key lesson this volume set out to teach.

---

## Volume 08 — What You Should Be Able to Do Now

- Explain, in plain language, what a forward pass, loss, and backpropagation each do
- Build and train a simple Keras `Sequential` model, and read its loss/validation curves to detect overfitting
- Explain why PyTorch's explicit training loop needs `zero_grad()`, `backward()`, and `step()`, in that order, every batch
- Describe what PyTorch Lightning and FastAI each abstract away, and when that abstraction is worth trading away raw control for
- Explain, given a new business problem, whether it calls for deep learning or gradient boosting — and justify the answer

**Next: `09_NLP_and_Generative_AI.md`**